<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/NLP_speech_to_text_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports and libraries**

In [27]:
!pip install -q -U transformers
!pip install -q scikit-learn
!pip install -q sentence-transformers
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [68]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata
from huggingface_hub import login
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import Callable
import os
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM,AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig
from millionaire_client.exceptions import GameError


# **Setup HuggingFace and Game APIs**

## **HuggingFace**

In [30]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

## **Game APIs**

In [31]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [32]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment']


In [33]:
from millionaire_client import MillionaireClient, AuthenticationError

In [34]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

In [35]:
API_URL = "http://131.175.15.22:51111/"
username = "GliEmbeddingRuspanti"
password = "GliEmbeddingRuspanti"

In [36]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, GliEmbeddingRuspanti! (Role: student)


# **Speech to text interface adaption**

In [2]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=55ac4936e5a7c59aef62426ac74df81f65d14206e1ed62c0c8c705ed676901fe
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [73]:
import whisper

'''
From documentation:
We have different options of models.
From the smallest to the largest:
Model | Size | English-only | Multilingual
tiny	  39 M	      ✓	              ✓
base	  74 M	      ✓	              ✓
small	  244 M	      ✓	              ✓
medium	769 M	      ✓	              ✓
large	  1550 M	    x	              ✓
turbo	  798 M	      x	              ✓
'''
audio_model = whisper.load_model("turbo")

In [74]:
def convert_to_indexes(answer):
  # Models we used are designed to get back a string representing the index of the correct answer.
  # Audio interface requires the model's answer to be A/B/C/D.
  # So we convert it through a dictionary.
  conversion = {
      '0': 'A',
      '1': 'B',
      '2': 'C',
      '3': 'D',
  }

  if answer in conversion.values():
    return answer

  if answer not in conversion.keys():
    raise ValueError(f"Invalid answer: {answer}\n answer type: {type(answer)}")

  return conversion.get(answer, answer)


In [6]:
import re

def clean_answer(answer):
  """
    Remove the prefix:
    'Option [A/B/C/D]' + character + space
  """
  answer = answer.strip()
  return re.sub(r'^Option\s+[A-D].?\s*', '', answer).strip()

In [75]:
from millionaire_client.models import Question
from millionaire_client.models import Option

'''
  print(f"\n--- Level {game.current_level} ---")
  print(f"Q: {question.text}")
  for opt in question.options:
      print(f"  [{opt.id}] {opt.text}")

  answer_summary, answer_input = answer_ensemble(
      models, question.text, options, system_prompt, verbose=verbose
  )
'''

def speech_to_text(current_level):
    current_level_prefix = f"level_{current_level}"

    question_audio = audio_model.transcribe(f"{current_level_prefix}_question.wav")

    question_text = clean_answer(question_audio["text"])

    options = []

    for i, letter in enumerate(['A', 'B', 'C', 'D']):
        audio = audio_model.transcribe(f"{current_level_prefix}_option_{letter}.wav")

        options.append(
            Option(
                id=i,
                text=clean_answer(audio["text"])
            )
        )

    question = Question(
        id=current_level,
        text=question_text,
        options=options,
        level=current_level
    )

    return question

# **Model classes**

In [37]:
class ModelFactory:
    """
    Base factory, subclasses create the pipeline
    create() produces Model instances sharing that pipeline.
    """
    def __init__(self, model_name: str, hf_token=None, device_map="cuda",
                 cache_dir=None, gen_args=None, quantization_config=None, trust_remote_code=True):
        self.model_name = model_name
        self.gen_args = gen_args or {}
        self._pipe = self._load_pipeline(
            model_name, hf_token, device_map, cache_dir, quantization_config, trust_remote_code
        )

    def _load_pipeline(self, model_name, hf_token, device_map, cache_dir, quantization_config, trust_remote_code):
        raise NotImplementedError

    def create(self, name: str, answer_fn: Callable, answers_in_question=True):
        """
        Produce a new Model instance sharing this factory's pipeline.
        Weights are not reloaded.
        """
        return HFPipelineModel(
            name=name,
            pipe=self._pipe,
            answer_fn=answer_fn,
            gen_args=self.gen_args,
            answers_in_question=answers_in_question,
        )


class HFCausalFactory(ModelFactory):
    """Factory for standard causal LMs (Llama, Phi, Qwen, ...)."""
    def _load_pipeline(self, model_name, hf_token, device_map: str = "cuda", cache_dir: str | None = None, quantization_config=None,trust_remote_code=False):
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=trust_remote_code, token=hf_token, cache_dir=cache_dir,
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        return pipeline("text-generation", model=model, tokenizer=tokenizer)


class HFSeq2SeqFactory(ModelFactory):
    """Factory for encoder-decoder models (Flan-T5, ...)."""
    def _load_pipeline(self, model_name, hf_token, device_map: str = "cuda", cache_dir: str | None = None, quantization_config=None):
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=True, token=hf_token, cache_dir=cache_dir,
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        return pipeline("text-generation", model=model, tokenizer=tokenizer)


class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"


class HFPipelineModel(Model):
    """
    A Model that uses a shared pipeline injected by a ModelFactory.
    Never loads weights itself — that is the factory's responsibility.
    """
    DEFAULT_GEN_ARGS = {
        "max_new_tokens": 600,
        "return_full_text": False,
        "temperature": 0.5,
        "do_sample": True,
    }

    def __init__(self, name: str, pipe, answer_fn: Callable[[str, dict], str],
                 gen_args: dict = None, answers_in_question: bool = True):
        super().__init__(name, answer_fn)
        self._pipe = pipe
        self.gen_args = {**self.DEFAULT_GEN_ARGS, **(gen_args or {})}
        self.answers_in_question = answers_in_question

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        """Generates and process the answer through answer_fn."""

        if self.answers_in_question:
          # Converte le opzioni in plain text
          options_text = "\n".join(
              [f"- {value}" for value in options.values()]
          )

          question_full = f"{question}\n\nPossible options:\n{options_text}"
        else:
          question_full = question

        raw_output = self.generate(question_full, system_prompt)
        print(f"MODEL ANSWER ----->{raw_output}")
        summary_answer, answer = self.answer_fn(raw_output, options)
        return summary_answer, answer

    def generate(self, question: str, system_prompt: str = "") -> str:
        prompt = f"{system_prompt}\n\nQuestion: {question}"

        output = self._pipe(prompt, **self.gen_args)
        return output[0]["generated_text"]

# **Answers logic implementation through functions**

Here we'll implement the logic behind the true answer we'll give to the game.

## **Regex direct extraction**

In [43]:
# Strategy 1: Regex-based direct extraction

def extract_by_regex(model_output: str):
    text = model_output.strip()

    summary = {'confidence': 1}

    m = re.search(r'(?:answer(?:\s+is)?|option|choice|select|pick|correct)\s*[:\-]?\s*\*{0,2}([ABCD])\*{0,2}', text, re.IGNORECASE)
    if m: return summary, m.group(1).upper()

    m = re.search(r'\b([ABCD])[\)\.:](?:\s|$)', text, re.IGNORECASE)
    if m: return summary, m.group(1).upper()
    # lone capital letter (models often deliberate, then conclude with the letter)
    letters = re.findall(r'(?<![a-zA-Z])([ABCD])(?![a-zA-Z])', text)
    if letters:
        return summary, letters[-1].upper()
    return summary, None

# Let's set up a small test to see if it works
test_cases = [
    ('The answer is A, because Napoleon was a political figure.', 'A'),
    ('Napoleon was a ruler. I think the answer is B.', 'B'),
    ('C) Un politico', 'C'),
    ('Napoleon era un generale. La risposta corretta e D.', 'D'),
    ('He was born in Corsica and rose to become emperor', None),
]
print('Regex extraction tests:')
for text, expected in test_cases:
    conf, result = extract_by_regex(text)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'  [{status}] Input: {repr(text[:55]):57s} -> Got: {result} [with confidence: {conf}], Expected: {expected}')

Regex extraction tests:
  [PASS] Input: 'The answer is A, because Napoleon was a political figur' -> Got: A [with confidence: {'confidence': 1}], Expected: A
  [PASS] Input: 'Napoleon was a ruler. I think the answer is B.'          -> Got: B [with confidence: {'confidence': 1}], Expected: B
  [PASS] Input: 'C) Un politico'                                          -> Got: C [with confidence: {'confidence': 1}], Expected: C
  [PASS] Input: 'Napoleon era un generale. La risposta corretta e D.'     -> Got: D [with confidence: {'confidence': 1}], Expected: D
  [PASS] Input: 'He was born in Corsica and rose to become emperor'       -> Got: None [with confidence: {'confidence': 1}], Expected: None


## **TF-IDF + cosine similarity**

In [44]:
# Strategy 2: TF-IDF + Cosine Similarity (Vector Space Model)

def pick_by_tfidf(model_output: str, options: dict):

    labels = list(options.keys())
    texts = [model_output] + [options[l] for l in labels]

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(texts)

    query_vec = tfidf_matrix[0]
    option_vecs = tfidf_matrix[1:]

    # cosine similarities
    scores = cosine_similarity(query_vec, option_vecs)[0]

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8

    normalized_margin = (
        (best_score - second_score) / score_range
        if len(labels) > 1 else 0.0
    )

    # --- probabilistic closeness of 2nd to 1st ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
        if len(labels) > 1 else 0.0
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}

model_response = "Napoleone era un grande leader politico e militare, imperatore dei francesi."

summary, best = pick_by_tfidf(model_response, options_test)

print("Picked option:", best)
print("summary:", summary)

Picked option: A
summary: {'best_option': 'A', 'best_score': 0.32858905992256504, 'best_probability': 0.3047505519470668, 'scores': {'A': 0.32858905992256504, 'B': 0.06962269809588174, 'C': 0.0, 'D': 0.09234056510965123}, 'softmax_probabilities': {'A': 0.3047505519470668, 'B': 0.23522140456331864, 'C': 0.21940174900740894, 'D': 0.24062629448220574}, 'gap_mean': 0.2746013055207207, 'relative_second_closeness': 0.4412110562772332, 'normalized_margin': 0.7189785553992648}


## **sBERT: A semantic similarity approach**

In [45]:
# Strategy 3: Sentence-BERT (sBERT) Semantic Similarity

sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def pick_by_sbert(model_output: str, options: dict):

    labels = list(options.keys())
    all_texts = [model_output] + [options[l] for l in labels]

    embeddings = sbert_model.encode(
        all_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_emb = embeddings[0]
    option_embs = embeddings[1:]

    # cosine similarity because embeddings are normalized
    scores = option_embs @ query_emb

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8

    normalized_margin = (
        (best_score - second_score) / score_range
        if len(labels) > 1 else 0.0
    )

    # --- probabilistic closeness of 2nd to 1st ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
        if len(labels) > 1 else 0.0
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'A politician',
    'B': 'a television personality',
    'C': 'something absurd',
    'D': 'a football player'
}

model_response = (
    "Napoleon was a great political and military leader, "
    "Emperor of the French."
)

summary, best = pick_by_sbert(model_response, options_test)

print("Picked option:", best)
print("Summary:", summary)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Picked option: A
Summary: {'best_option': 'A', 'best_score': 0.3144106864929199, 'best_probability': 0.3078668713569641, 'scores': {'A': 0.3144106864929199, 'B': 0.006208924576640129, 'C': -0.07364878058433533, 'D': 0.13410882651805878}, 'softmax_probabilities': {'A': 0.3078668713569641, 'B': 0.22621044516563416, 'C': 0.2088482528924942, 'D': 0.25707441568374634}, 'gap_mean': 0.2921876907348633, 'relative_second_closeness': 0.45504625162805523, 'normalized_margin': 0.4646243155002594}


## **Cross encoder**

In [46]:
modelCrossencoder = CrossEncoder('cross-encoder/stsb-distilroberta-base', trust_remote_code=True)

'''
def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    roberta_inputs = [[model_output, options[l]] for l in labels]
    scores = modelCrossencoder.predict(roberta_inputs)
    best_label = labels[int(np.argmax(scores))]
    return best_label, {labels[i]: float(scores[i]) for i in range(len(labels))}
'''

def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    pairs = [[model_output, options[l]] for l in labels]

    scores = modelCrossencoder.predict(pairs)

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = float(best_score - np.mean(sorted_scores[1:])) if len(labels) > 1 else 0.0

    # --- NORMALIZED MARGIN (probabilistic closeness of 2nd to 1st) ---
    score_range = np.max(scores) - np.min(scores) + 1e-8
    normalized_margin = (best_score - second_score) / score_range

    # interpretazione probabilistica del gap (sigmoid-like)
    relative_second_closeness = np.exp(second_score) / (np.exp(best_score) + np.exp(second_score))

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i]) for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i]) for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'a television personality',
    'B': 'A politician',
    'C': 'something absurd',
    'D': 'a football player'
}

model_response = (
    "Napoleon was a great political and military leader, "
    "Emperor of the French."
)

summary, best = pick_by_crossencoder(model_response, options_test)

print("Picked option:", best)
print("Summary", summary)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Picked option: B
Summary {'best_option': 'B', 'best_score': 0.2227260023355484, 'scores': {'A': 0.028937358409166336, 'B': 0.2227260023355484, 'C': 0.0470624640583992, 'D': 0.014161717146635056}, 'softmax_probabilities': {'A': 0.23710934817790985, 'B': 0.2878127694129944, 'C': 0.24144618213176727, 'D': 0.2336316853761673}, 'gap_mean': 0.19267214834690094, 'relative_second_closeness': 0.45619669656547107, 'normalized_margin': 0.842251181602478}


# **Models**

In [85]:
llama_factory = HFCausalFactory(
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
)

llama_sbert        = llama_factory.create("llama-sbert",        pick_by_sbert)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [51]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

gemma_factory = HFCausalFactory(
    model_name="google/gemma-2-9b-it",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    trust_remote_code=False,
)

gemma_model = gemma_factory.create("gemma2-9b-crossencoder", pick_by_crossencoder)

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

# **Speech interaction**

## **Models definition with prompts**

In [86]:
models = {
    "llama model": {
        "model": [llama_sbert],
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },
}

In [87]:
def answer_ensemble(models: list, question_text: str, options: dict, system_prompt: str, verbose=False):
    """
    Chiama model.answer su ogni modello dell'array e restituisce
    la risposta con la best_score più alta tra tutti i modelli.
    """
    best_answer = None
    best_summary = None
    best_score = -float('inf')

    for model in models:
        summary, answer = model.answer(question_text, options, system_prompt)
        score = summary.get("best_score", 0.0) if isinstance(summary, dict) else 0.0

        if verbose:
            print(f"  [{model.name}] answer={answer} | best_score={score:.4f}")

        if score > best_score:
            best_score = score
            best_answer = answer
            best_summary = summary

    if verbose:
        print(f"  => Ensemble winner: answer={best_answer} | score={best_score:.4f}")

    return best_summary, best_answer

In [78]:
from IPython.display import Audio, display

In [79]:
comp_id = 1

In [80]:
def save_audio(data: bytes, filename: str):
    """Save audio bytes to a WAV file."""
    with open(filename, "wb") as f:
        f.write(data)
    print(f"  Saved: {filename}")

In [81]:
def play_game(game, models_configs, system_prompt):
    log = []

    while game.in_progress:
        # Fetch question audio
        print("Fetching question audio...")
        try:
            question_audio = game.fetch_audio_question()
            save_audio(question_audio, f"level_{game.current_level}_question.wav")
            display(Audio(question_audio))
        except GameError as e:
            print(f"Error fetching question audio: {e}")
            break

        # Fetch option audios sequentially (A, B, C, D)
        option_map = {}
        audios = []
        for i in range(4):
            letter = chr(65 + i)  # A, B, C, D
            print(f"Fetching option {letter} audio...")
            try:
                option_audio = game.fetch_audio_option_next()
                save_audio(option_audio, f"level_{game.current_level}_option_{letter}.wav")
                audios.append(option_audio)
            except GameError as e:
                print(f"Error fetching option audio: {e}")
                break

            # Store mapping for answer submission
            if game.current_question and i < len(game.current_question.options):
                option_map[letter] = game.current_question.options[i].id

        display(Audio(audios[0]),
                Audio(audios[1]),
                Audio(audios[2]),
                Audio(audios[3])) # Pardon this horrific hard-coded stuff, due to (even more horrific) ipykernel logic. ~Federico

        if not option_map:
            print("No question available. Game may have ended.")
            break

        game.refresh_state()

        question_from_audio = speech_to_text(game.current_level)
        print(f"\n--- Level {game.current_level} (QUESTION FROM AUDIO) ---")
        print(f"Q: {question_from_audio.text}")
        print()

        for opt in question_from_audio.options:
            print(f"  [{opt.id}] {opt.text}")

        # Get time remaining (timer starts after Option D is fetched)
        time_left = game.time_remaining
        if time_left is not None:
            print(f"\nTime remaining: {time_left:.1f}s")

        options = {f"{opt.id}": opt.text for opt in question_from_audio.options}

        t0 = time.time()
        answer_summary, answer_input = answer_ensemble(
            models, question_from_audio.text, options, system_prompt, verbose=False
        )
        inference_time = time.time() - t0

        answer_id = int(answer_input)
        choosen_answer = question_from_audio.options[answer_id]
        result = game.answer(answer_id)

        # Handle timeout
        if result.timed_out or result.status == "timeout":
            print(" TIME'S UP!")
            print(f"\n Game Over! You ran out of time.")
            print(f" Final earnings: ${result.earned_amount:,.2f}")
            break

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        else:
            print(" WRONG!")
            print(f"\n Game Over!")
            print(f" Final earnings: ${result.earned_amount:,.2f}")

        entry = {
            'level'          : game.current_level,
            'question'       : question_from_audio.text,
            'options'        : question_from_audio.options,
            'chosen_option'  : choosen_answer.text,
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            'answer_summary' : answer_summary,
        }
        log.append(entry)

    ensemble_name = " + ".join(m.name for m in models)

    summary = {
        'model'          : ensemble_name,
        'final_level'    : game.current_level,
        'earned_amount'  : game.earned_amount,
        'num_questions'  : len(log),
        'num_correct'    : sum(1 for e in log if e['correct']),
        'num_timed_out'  : sum(1 for e in log if e['timed_out']),
        'avg_inference_s': round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'            : log,
    }
    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [88]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")

    models = config["model"] # lista di modelli
    system_prompt = config["system_prompt"] # prompt

    print(models)

    model_results = []

    for comp_id in [0, 1, 2, 3]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id, mode="speech")

        summary = play_game(game, models, system_prompt)

        model_results.append(summary)

    results[model_name] = model_results


########## MODEL: llama model ##########
[HFPipelineModel(name='llama-sbert', answer_fn='pick_by_sbert')]

--- Competition 0 ---
Fetching question audio...
  Saved: level_1_question.wav


Fetching option A audio...
  Saved: level_1_option_A.wav
Fetching option B audio...
  Saved: level_1_option_B.wav
Fetching option C audio...
  Saved: level_1_option_C.wav
Fetching option D audio...
  Saved: level_1_option_D.wav


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 (QUESTION FROM AUDIO) ---
Q: What is the fundamental principle of Paul McCartney's approach to songwriting and music creation, as described in the article?

  [0] strict adherence to classical music forms.
  [1] versatility and exploration across various genres.
  [2] focusing solely on rock and roll.
  [3] limited involvement in the recording process.

Time remaining: 26.3s


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 (QUESTION FROM AUDIO) ---
Q: kendall studied the ways in which human body systems work together he compared the respiratory and circulatory systems in which way are these two systems similar to each other

  [0] they both send messages to the body.
  [1] they both pump blood through the body.
  [2] They both digest nutrients for the body.
  [3] they both bring oxygen to the body.

Time remaining: 26.6s
MODEL ANSWER ----->
 WRONG!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00

--- Competition 3 ---
Fetching question audio...
  Saved: level_1_question.wav


Fetching option A audio...
  Saved: level_1_option_A.wav
Fetching option B audio...
  Saved: level_1_option_B.wav
Fetching option C audio...
  Saved: level_1_option_C.wav
Fetching option D audio...
  Saved: level_1_option_D.wav


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 (QUESTION FROM AUDIO) ---
Q: What is the minimum value of the expression x plus 4z as a function defined on R3, subject to the constraint x squared plus y fit plus silt plus sid com s, the sto so b cub db down d 1 d soar of.

  [0] zero, zero.
  [1] square Duplo 3434
  [2] Heh heh, savo pinaan! Tu will? For said bud, so come wheeze, we're J's, twee smart!
  [3] Carnify... The fftf ff. Fff. Ff. Make... Sure it is.

Time remaining: 17.9s
MODEL ANSWER -----> The. Fff... The. Fff... Fff.
- The. Fff. Ff. Fff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. Ff. F

In speech mode, the 30 seconds timer is starting when we are requesting the last option!!

In [84]:
# Show leaderboard position (speech mode)
lb = client.leaderboard.get(competition_id=comp_id, limit=10, mode="speech") # <---- include mode!!
print(f"\n=== Speech Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Speech Leaderboard for Maths ===
  1. Zero37: $4,000.00 (Level 7)
  2. grepapetti: $4,000.00 (Level 7)
  3. leonfuss: $1,000.00 (Level 5)
  4. ciani: $500.00 (Level 4)
  5. chrisgpt: $500.00 (Level 4)
